# Delta Live Tables - CDC (Change Data Capture)

---

## 📊 O que é CDC?

**CDC (Change Data Capture)** é o processo de capturar e processar **mudanças** em dados:
- **Inserts** (novos registros)
- **Updates** (alterações)
- **Deletes** (exclusões)

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Source System  │────▶│   CDC Feed      │────▶│  DLT Pipeline   │
│  (OLTP Database)│     │  (Changes Only) │     │  (Target Table) │
└─────────────────┘     └─────────────────┘     └─────────────────┘
         │                      │                       │
    Transações             INSERT                  Tabela Delta
    Completas              UPDATE                  Atualizada
                           DELETE
```

---

## 🔄 APPLY CHANGES INTO

O comando `APPLY CHANGES INTO` processa um feed CDC e aplica as mudanças em uma tabela de destino.

### Componentes Essenciais

| Componente | Descrição | Obrigatório |
|------------|-----------|-------------|
| **Target** | Tabela de destino | ✅ Sim |
| **Source** | Streaming table com dados CDC | ✅ Sim |
| **KEYS** | Coluna(s) para identificar registro único | ✅ Sim |
| **SEQUENCE BY** | Coluna para ordenar mudanças | ✅ Sim |
| **COLUMNS** | Colunas a incluir | Opcional |
| **STORED AS** | SCD Type 1 ou Type 2 | Opcional |

---

## 📋 Sintaxe SQL

### Estrutura Básica

```sql
-- 1. Criar a tabela de destino (vazia)
CREATE OR REFRESH STREAMING TABLE target_table;

-- 2. Aplicar mudanças do CDC feed
APPLY CHANGES INTO LIVE.target_table
FROM STREAM(LIVE.source_cdc)
KEYS (primary_key_column)
SEQUENCE BY sequence_column
```

### Exemplo Completo

```sql
-- Tabela de destino para clientes
CREATE OR REFRESH STREAMING TABLE customers;

-- Aplicar mudanças CDC
APPLY CHANGES INTO LIVE.customers
FROM STREAM(LIVE.customers_cdc)
KEYS (customer_id)
SEQUENCE BY updated_at
COLUMNS * EXCEPT (_rescued_data, _metadata)
STORED AS SCD TYPE 1
```

---

## 🐍 Sintaxe Python

### Estrutura Básica

```python
import dlt

# 1. Criar tabela de destino
dlt.create_streaming_table("target_table")

# 2. Aplicar mudanças
dlt.apply_changes(
    target="target_table",
    source="source_cdc",
    keys=["primary_key_column"],
    sequence_by="sequence_column"
)
```

### Exemplo Completo

```python
import dlt

# Tabela de destino
dlt.create_streaming_table(
    name="customers",
    comment="Customer master data with CDC"
)

# Aplicar mudanças CDC
dlt.apply_changes(
    target="customers",
    source="customers_cdc",
    keys=["customer_id"],
    sequence_by="updated_at",
    except_column_list=["_rescued_data", "_metadata"],
    stored_as_scd_type=1
)
```

---

## 📚 SCD Types (Slowly Changing Dimensions)

### SCD Type 1 - Sobrescrever

**Comportamento:** Atualiza o registro existente, **sem manter histórico**.

```sql
APPLY CHANGES INTO LIVE.customers
FROM STREAM(LIVE.customers_cdc)
KEYS (customer_id)
SEQUENCE BY updated_at
STORED AS SCD TYPE 1  -- ou simplesmente omita (é o padrão)
```

| customer_id | name | city | updated_at |
|-------------|------|------|------------|
| 1 | João | São Paulo | 2024-01-15 |

> ⚠️ **SCD Type 1** é o **padrão** quando não especificado.

---

### SCD Type 2 - Manter Histórico

**Comportamento:** Cria nova versão do registro, **mantendo histórico completo**.

```sql
APPLY CHANGES INTO LIVE.customers
FROM STREAM(LIVE.customers_cdc)
KEYS (customer_id)
SEQUENCE BY updated_at
STORED AS SCD TYPE 2
```

Colunas adicionadas automaticamente:

| Coluna | Descrição |
|--------|-----------|
| `__START_AT` | Início da validade |
| `__END_AT` | Fim da validade (null = atual) |

**Resultado:**

| customer_id | name | city | __START_AT | __END_AT |
|-------------|------|------|------------|----------|
| 1 | João | Rio | 2024-01-01 | 2024-01-15 |
| 1 | João | São Paulo | 2024-01-15 | null |

---

## 🔑 KEYS e SEQUENCE BY

### KEYS - Identificador Único

Define qual(is) coluna(s) identificam um registro único.

```sql
-- Chave simples
KEYS (customer_id)

-- Chave composta
KEYS (order_id, product_id)
```

### SEQUENCE BY - Ordenação de Mudanças

Define como ordenar múltiplas mudanças para o mesmo registro.

```sql
-- Por timestamp
SEQUENCE BY updated_at

-- Por número de sequência
SEQUENCE BY change_sequence_number

-- Por tempo de ingestão
SEQUENCE BY ingestion_timestamp
```

> 💡 **Importante:** Se chegarem múltiplas mudanças para a mesma KEY, o SEQUENCE BY determina qual é a mais recente.

---

## 🔧 Opções Adicionais

### COLUMNS - Selecionar Colunas

```sql
-- Todas as colunas
COLUMNS *

-- Excluir colunas específicas
COLUMNS * EXCEPT (col1, col2)

-- Selecionar colunas específicas
COLUMNS (customer_id, name, email, updated_at)
```

### Python - Parâmetros Equivalentes

```python
dlt.apply_changes(
    target="table",
    source="source",
    keys=["id"],
    sequence_by="timestamp",
    
    # Excluir colunas
    except_column_list=["_rescued_data"],
    
    # OU incluir apenas específicas
    column_list=["id", "name", "email"],
    
    # Ignorar nulos no sequence
    ignore_null_updates=True,
    
    # Aplicar como delete se condição for true
    apply_as_deletes=expr("operation = 'DELETE'"),
    
    # Aplicar como truncate
    apply_as_truncates=expr("operation = 'TRUNCATE'")
)
```

---

## ✅ Checklist - CDC

### Conceitos
- [ ] CDC captura **INSERT**, **UPDATE** e **DELETE**
- [ ] `APPLY CHANGES INTO` processa feed CDC
- [ ] **KEYS** identifica registro único
- [ ] **SEQUENCE BY** ordena mudanças (timestamp, sequence number)

### SCD Types
- [ ] **SCD Type 1** = sobrescreve, **sem histórico** (padrão)
- [ ] **SCD Type 2** = mantém histórico com `__START_AT` e `__END_AT`
- [ ] Se não especificar, usa **SCD Type 1**

### Sintaxe SQL
- [ ] `CREATE OR REFRESH STREAMING TABLE target;`
- [ ] `APPLY CHANGES INTO LIVE.target`
- [ ] `FROM STREAM(LIVE.source)`
- [ ] `KEYS (column)`
- [ ] `SEQUENCE BY column`

### Sintaxe Python
- [ ] `dlt.create_streaming_table("target")`
- [ ] `dlt.apply_changes(target=, source=, keys=[], sequence_by=)`
- [ ] `stored_as_scd_type=1` ou `stored_as_scd_type=2`
- [ ] `except_column_list=[]` para excluir colunas

---

## 🔗 Referências

- [CDC com APPLY CHANGES](https://learn.microsoft.com/en-us/azure/databricks/delta-live-tables/cdc)
- [SCD no Delta Live Tables](https://learn.microsoft.com/en-us/azure/databricks/delta-live-tables/scd)
- [Referência Python DLT](https://learn.microsoft.com/en-us/azure/databricks/delta-live-tables/python-ref)
- [O que é CDC?](https://learn.microsoft.com/pt-br/azure/databricks/ldp/what-is-change-data-capture)